# 🎯 Customer Intelligence Video Analytics Pipeline

This notebook runs the complete restaurant customer intelligence pipeline in **Google Colab** or **Kaggle**.
It uses your custom-trained YOLO model to detect and classify guests vs. staff, tracks them with clean sequential IDs (#1, #2...), and overlays a live analytics dashboard on the video.

---

## 🚀 Setup Instructions
1. **Runtime**: Go to `Runtime → Change runtime type → T4 GPU`
2. **Files needed**: Upload both of the following to the **file explorer** (📁 icon on the left):
   - `Dark_lighting.mp4` — your test video
   - `yolo_staff_customer.pt` — your custom YOLO model

> **Tip**: If files are on Google Drive, run the optional mount cell below first.

In [ ]:
# ============================================================
# OPTIONAL: Mount Google Drive if your files are stored there
# ============================================================
# Uncomment and run this cell if you want to load files from Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# Then update the paths below:
# VIDEO_PATH = "/content/drive/MyDrive/CustomerIntel/Dark_lighting.mp4"
# MODEL_PATH = "/content/drive/MyDrive/CustomerIntel/yolo_staff_customer.pt"

# Default paths (for files uploaded directly to Colab)
VIDEO_PATH = "Dark_lighting.mp4"
MODEL_PATH = "yolo_staff_customer.pt"
OUTPUT_PATH = "output_dark_lighting.mp4"

print(f"Video:  {VIDEO_PATH}")
print(f"Model:  {MODEL_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# ============================================================
# Install dependencies
# ============================================================
!pip install -q ultralytics opencv-python-headless

## 🧠 Core Classes: Sequential Tracker & Frame Streamer

In [ ]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO


class SequentialPositionTracker:
    """
    Centroid-based multi-object tracker that assigns clean sequential integer IDs (#1, #2...).
    Tracks are expired if unseen for more than max_missing_frames frames.
    """
    def __init__(self, max_distance=180, max_missing_frames=120):
        self.tracks = {}
        self.max_distance = max_distance
        self.max_missing_frames = max_missing_frames
        self.next_id = 1

    def _centroid(self, bbox):
        x1, y1, x2, y2 = bbox
        return ((x1 + x2) / 2, (y1 + y2) / 2)

    def update(self, detections, frame_id, ts):
        results = []
        matched_tokens = set()

        for bbox in detections:
            centroid = self._centroid(bbox)
            best_token, best_dist = None, float('inf')

            for token, track in self.tracks.items():
                if token in matched_tokens:
                    continue
                dist = np.sqrt(
                    (centroid[0] - track['centroid'][0]) ** 2 +
                    (centroid[1] - track['centroid'][1]) ** 2
                )
                if dist < best_dist:
                    best_dist = dist
                    best_token = token

            if best_token and best_dist < self.max_distance:
                self.tracks[best_token].update({
                    'centroid': centroid, 'bbox': bbox,
                    'last_frame': frame_id, 'last_ts': ts
                })
                matched_tokens.add(best_token)
                results.append((best_token, bbox, False))
            else:
                token = self.next_id
                self.next_id += 1
                self.tracks[token] = {
                    'centroid': centroid, 'bbox': bbox,
                    'last_frame': frame_id, 'entry_ts': ts,
                    'last_ts': ts
                }
                matched_tokens.add(token)
                results.append((token, bbox, True))

        return results

    def get_exited(self, frame_id):
        """Remove tracks not seen for max_missing_frames — prevents ID hijacking by new detections."""
        exited = []
        for token, track in list(self.tracks.items()):
            if (frame_id - track['last_frame']) > self.max_missing_frames:
                exited.append((token, track['entry_ts'], track['last_ts']))
                del self.tracks[token]
        return exited


def apply_clahe(frame):
    """Apply CLAHE contrast normalization — improves detection in dim restaurant lighting."""
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    enhanced = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


def stream_frames(source, fps_target=8):
    """Yield (frame_id, timestamp_sec, frame) at the target FPS, applying CLAHE enhancement."""
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {source}")
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    skip = max(1, int(native_fps / fps_target))

    frame_id = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % skip == 0:
            ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            yield frame_id, ts, apply_clahe(frame)
        frame_id += 1
    cap.release()


print("✅ Classes defined. Ready to run.")

## 🎥 Run the Full Pipeline

In [ ]:
def process_pipeline(video_path=VIDEO_PATH, output_path=OUTPUT_PATH, model_path=MODEL_PATH, fps_target=8):
    print("Loading YOLO model...")
    detector = YOLO(model_path)

    # Auto-select best available device: CUDA > MPS > CPU
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    print(f"Using device: {device}")

    tracker = SequentialPositionTracker(max_distance=180, max_missing_frames=120)
    token_class_votes = {}  # ID -> list of detected class indices
    token_final_class = {}  # ID -> "guest" or "staff"
    cumulative_guests = set()  # All unique guest IDs ever seen
    cumulative_staff = set()   # All unique staff IDs ever seen

    writer = None
    frame_count = 0

    # Color palette
    COLOR_GUEST = (0, 165, 255)   # Orange for guests
    COLOR_STAFF = (0, 255, 0)     # Green for staff
    COLOR_BG    = (30, 30, 30)    # Dark background for stats card

    print(f"\nProcessing {video_path} ...")
    for fid, ts, frame in stream_frames(video_path, fps_target=fps_target):
        if writer is None:
            h, w = frame.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps_target, (w, h))

        # ── Detection ───────────────────────────────────────────
        results = detector(frame, conf=0.25, verbose=False, device=device)
        bboxes, box_classes = [], []
        for box in results[0].boxes:
            bboxes.append(box.xyxy[0].tolist())
            box_classes.append(int(box.cls[0]))

        # ── Tracking ─────────────────────────────────────────────
        tracks = tracker.update(bboxes, fid, ts)
        tracker.get_exited(fid)  # Expire lost tracks to avoid ID hijacking

        # ── Majority-vote role classification ────────────────────
        for i, (token, bbox, is_new) in enumerate(tracks):
            cls = box_classes[i]
            token_class_votes.setdefault(token, []).append(cls)
            votes = token_class_votes[token]
            is_staff = votes.count(1) > votes.count(0)  # cls 1 = staff, 0 = customer
            token_final_class[token] = "staff" if is_staff else "guest"

            if is_staff:
                cumulative_staff.add(token)
                cumulative_guests.discard(token)  # Correct if initially misclassified
            elif token not in cumulative_staff:
                cumulative_guests.add(token)

        # ── Draw bounding boxes & labels ─────────────────────────
        for token, bbox, _ in tracks:
            role = token_final_class.get(token, "guest")
            x1, y1, x2, y2 = [int(v) for v in bbox]
            label = f"#{token}{role}"  # e.g. "#1guest" or "#6staff"
            color = COLOR_STAFF if role == "staff" else COLOR_GUEST

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            (tw, _), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, max(y1 - 20, 0)), (x1 + tw + 6, max(y1 - 2, 0)), color, -1)
            cv2.putText(frame, label, (x1 + 3, max(y1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

        # ── Live Dashboard Card (top-left) ───────────────────────
        n_covers  = len(cumulative_guests)
        n_staff   = len(cumulative_staff)
        a_guests  = sum(1 for t, _, _ in tracks if token_final_class.get(t) == "guest")
        a_staff   = sum(1 for t, _, _ in tracks if token_final_class.get(t) == "staff")

        cv2.rectangle(frame, (10, 10), (440, 115), COLOR_BG, -1)
        cv2.rectangle(frame, (10, 10), (440, 115), (100, 100, 100), 2)
        cv2.putText(frame, "CUSTOMER INTELLIGENCE — LIVE SCAN",
                    (20, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 200, 255), 1, cv2.LINE_AA)
        cv2.putText(frame, f"Covers: {n_covers}   Staff: {n_staff}",
                    (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"In frame: {a_guests} guest(s), {a_staff} staff",
                    (20, 83), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)
        cv2.putText(frame, f"t={ts:.1f}s  frame={fid}",
                    (20, 103), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (140, 140, 140), 1, cv2.LINE_AA)

        writer.write(frame)
        frame_count += 1
        if frame_count % 100 == 0:
            print(f"  [{frame_count:4d} frames | {ts:5.1f}s]  Covers: {n_covers}  Staff: {n_staff}")

    if writer:
        writer.release()
    print(f"\n✅ Done. {frame_count} frames written to: {output_path}")
    print(f"   Final → Covers (guests): {len(cumulative_guests)}   Staff: {len(cumulative_staff)}")
    return output_path


output = process_pipeline()

## 📥 Download the Annotated Output Video

In [ ]:
# ============================================================
# Download the output video to your local machine
# ============================================================
try:
    from google.colab import files
    print(f"Downloading {output} ...")
    files.download(output)
except ImportError:
    # Running on Kaggle — output is already in /kaggle/working/
    print(f"Kaggle detected. Output saved to: /kaggle/working/{output}")
    print("You can download it from the Output tab on the right side.")

---

## 📊 What to Look For in the Output Video

| Visual Element | Meaning |
|---|---|
| 🟩 Green box + `#Nstaff` | Staff member — ID N |
| 🟧 Orange box + `#Nguest` | Guest / Customer — ID N |
| Dashboard Card (top-left) | Live count of unique Covers & Staff |

### ⚠️ Known Failure Modes in Dark Lighting

| Failure | Root Cause | Impact |
|---|---|---|
| **Missed detections** | YOLO confidence drops below 0.25 in very dark areas | Some guests not counted |
| **ID fragmentation** | Track expires after 15s if person disappears in shadow, gets new ID on re-detection | Over-counting unique guests |
| **Ghost tracks** | Low-light noise creates 1-frame false detections | Short-lived IDs (filter by duration > 2s) |
| **Role flip** | Dark clothing makes staff look like guests initially | Stabilises after ~5 frames via majority vote |